In [1]:
!pip install meteostat holidays pandas==2.2.2 numpy

  Using cached meteostat-2.1.3-py3-none-any.whl.metadata (5.2 kB)
INFO: pip is looking at multiple versions of meteostat to determine which version is compatible with other requirements. This could take a while.
  Using cached meteostat-2.1.2-py3-none-any.whl.metadata (5.0 kB)
  Using cached meteostat-2.1.1-py3-none-any.whl.metadata (5.0 kB)
  Using cached meteostat-2.1.0-py3-none-any.whl.metadata (5.0 kB)
  Using cached meteostat-2.0.1-py3-none-any.whl.metadata (5.0 kB)
  Using cached meteostat-2.0.0-py3-none-any.whl.metadata (5.3 kB)
  Using cached meteostat-1.7.6-py3-none-any.whl.metadata (4.6 kB)
Using cached meteostat-1.7.6-py3-none-any.whl (33 kB)


In [2]:
import pandas as pd
import numpy as np
from meteostat import Point, Hourly
from datetime import datetime
import holidays

In [4]:
import pandas as pd

df = pd.read_csv('raw_data/iso_ne_load_2018_2025.csv')

df['datetime'] = pd.to_datetime(df['datetime'])
df.set_index('datetime', inplace=True)

df = df.sort_index()

df.head()

,load
datetime,
2018-01-01 00:00:00,15760
2018-01-01 00:00:00,17762
2018-01-01 01:00:00,17644
2018-01-01 01:00:00,15297
2018-01-01 02:00:00,17617


In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 70128 entries, 2018-01-01 00:00:00 to 2025-12-31 23:00:00
Data columns (total 1 columns):
 #   Column  Non-Null Count  Dtype
---  ------  --------------  -----
 0   load    70128 non-null  int64
dtypes: int64(1)
memory usage: 1.1 MB


In [6]:
import requests
import pandas as pd

def fetch_weather_openmeteo(lat, lon, start_date, end_date):

    url = "https://archive-api.open-meteo.com/v1/archive"

    params = {
        "latitude": lat,
        "longitude": lon,
        "start_date": start_date,
        "end_date": end_date,
        "hourly": "temperature_2m,relative_humidity_2m,precipitation,windspeed_10m",
        "timezone": "UTC"
    }

    response = requests.get(url, params=params)
    data = response.json()

    df = pd.DataFrame(data["hourly"])
    df["time"] = pd.to_datetime(df["time"])
    df.set_index("time", inplace=True)

    return df

In [7]:
stations = {
    "Boston": (42.3601, -71.0589),
    "Hartford": (41.7658, -72.6734),
    "Providence": (41.8240, -71.4128),
    "Portland": (43.6591, -70.2568),
    "Burlington": (44.4759, -73.2121)
}

weather_data = []

for city, (lat, lon) in stations.items():

    data = fetch_weather_openmeteo(
        lat, lon,
        "2018-01-01",
        "2025-12-31"
    )

    data.rename(columns={
        "temperature_2m": f"Temp_{city}",
        "relative_humidity_2m": f"Humidity_{city}",
        "windspeed_10m": f"Wind_{city}",
        "precipitation": f"Precip_{city}"
    }, inplace=True)

    weather_data.append(data)

weather = pd.concat(weather_data, axis=1)

weather.head()

,Temp_Boston,Humidity_Boston,Precip_Boston,Wind_Boston,Temp_Hartford,Humidity_Hartford,Precip_Hartford,Wind_Hartford,Temp_Providence,Humidity_Providence,Precip_Providence,Wind_Providence,Temp_Portland,Humidity_Portland,Precip_Portland,Wind_Portland,Temp_Burlington,Humidity_Burlington,Precip_Burlington,Wind_Burlington
time,,,,,,,,,,,,,,,,,,,,
2018-01-01 00:00:00,-15.0,57,0.0,17.9,-15.5,58,0.0,11.2,-14.3,51,0.0,15.8,-18.6,67,0.0,5.9,-17.7,67,0.0,19.3
2018-01-01 01:00:00,-15.9,58,0.0,16.5,-16.6,60,0.0,8.1,-14.8,53,0.0,16.8,-22.1,72,0.0,10.0,-18.2,67,0.0,18.3
2018-01-01 02:00:00,-16.6,59,0.0,14.5,-16.9,60,0.0,8.7,-15.2,54,0.0,16.1,-22.8,73,0.0,9.9,-18.3,66,0.0,18.1
2018-01-01 03:00:00,-17.3,61,0.0,13.8,-16.8,62,0.0,10.1,-15.8,55,0.0,16.4,-24.3,76,0.0,10.9,-18.4,66,0.0,17.6
2018-01-01 04:00:00,-17.9,62,0.0,14.4,-17.2,65,0.0,10.2,-16.4,56,0.0,16.2,-25.1,76,0.0,10.7,-18.4,66,0.0,16.5


In [8]:
df = df.join(weather, how="inner")
df.interpolate(method="time", inplace=True)
df.dropna(inplace=True)

df.head()

,load,Temp_Boston,Humidity_Boston,Precip_Boston,Wind_Boston,Temp_Hartford,Humidity_Hartford,Precip_Hartford,Wind_Hartford,Temp_Providence,...,Precip_Providence,Wind_Providence,Temp_Portland,Humidity_Portland,Precip_Portland,Wind_Portland,Temp_Burlington,Humidity_Burlington,Precip_Burlington,Wind_Burlington
2018-01-01 00:00:00,15760,-15.0,57,0.0,17.9,-15.5,58,0.0,11.2,-14.3,...,0.0,15.8,-18.6,67,0.0,5.9,-17.7,67,0.0,19.3
2018-01-01 00:00:00,17762,-15.0,57,0.0,17.9,-15.5,58,0.0,11.2,-14.3,...,0.0,15.8,-18.6,67,0.0,5.9,-17.7,67,0.0,19.3
2018-01-01 01:00:00,17644,-15.9,58,0.0,16.5,-16.6,60,0.0,8.1,-14.8,...,0.0,16.8,-22.1,72,0.0,10.0,-18.2,67,0.0,18.3
2018-01-01 01:00:00,15297,-15.9,58,0.0,16.5,-16.6,60,0.0,8.1,-14.8,...,0.0,16.8,-22.1,72,0.0,10.0,-18.2,67,0.0,18.3
2018-01-01 02:00:00,17617,-16.6,59,0.0,14.5,-16.9,60,0.0,8.7,-15.2,...,0.0,16.1,-22.8,73,0.0,9.9,-18.3,66,0.0,18.1


In [9]:
import numpy as np

# Basic time features
df["Hour"] = df.index.hour
df["DayOfWeek"] = df.index.dayofweek
df["Month"] = df.index.month

# Weekend flag
df["Weekend"] = df["DayOfWeek"].apply(lambda x: 1 if x >= 5 else 0)

In [10]:
# Hour
df["Hour_sin"] = np.sin(2 * np.pi * df["Hour"] / 24)
df["Hour_cos"] = np.cos(2 * np.pi * df["Hour"] / 24)

# Day of week
df["Day_sin"] = np.sin(2 * np.pi * df["DayOfWeek"] / 7)
df["Day_cos"] = np.cos(2 * np.pi * df["DayOfWeek"] / 7)

# Month
df["Month_sin"] = np.sin(2 * np.pi * df["Month"] / 12)
df["Month_cos"] = np.cos(2 * np.pi * df["Month"] / 12)

In [11]:
import holidays

us_holidays = holidays.US()

df["Holiday"] = df.index.date
df["Holiday"] = df["Holiday"].apply(lambda x: 1 if x in us_holidays else 0)

In [12]:
base_temp = 18  # Celsius base

for city in ["Boston", "Hartford", "Providence", "Portland", "Burlington"]:

    df[f"CDH_{city}"] = np.maximum(df[f"Temp_{city}"] - base_temp, 0)
    df[f"HDH_{city}"] = np.maximum(base_temp - df[f"Temp_{city}"], 0)

In [13]:
df["rolling_24"] = df["load"].rolling(24).mean()
df["rolling_168"] = df["load"].rolling(168).mean()

In [14]:
df.interpolate(method="time", inplace=True)
df.dropna(inplace=True)

df.shape

(69961, 44)

In [15]:
df.isna().sum().sum()

np.int64(0)

In [16]:
df.describe().T.head()

,count,mean,std,min,25%,50%,75%,max
load,69961.0,14710.854262,11964.083934,0.0,11711.0,13626.0,15678.0,99991.0
Temp_Boston,69961.0,11.863563,10.059707,-22.5,3.7,11.8,20.0,38.8
Humidity_Boston,69961.0,68.664413,18.932531,10.0,54.0,70.0,85.0,100.0
Precip_Boston,69961.0,0.142579,0.631828,0.0,0.0,0.0,0.0,16.9
Wind_Boston,69961.0,13.749432,6.520277,0.0,8.9,12.8,17.7,49.8


In [24]:
df.columns.values[0] = 'load'
print(df)

                      load  Temp_Boston  Humidity_Boston  Precip_Boston  \
2018-01-07 23:00:00  15542        -15.1               63            0.0   
2018-01-08 00:00:00  17627        -14.0               60            0.0   
2018-01-08 00:00:00  14818        -14.0               60            0.0   
2018-01-08 01:00:00  17553        -13.2               60            0.0   
2018-01-08 01:00:00  14408        -13.2               60            0.0   
...                    ...          ...              ...            ...   
2025-12-31 21:00:00  15850         -1.4               49            0.0   
2025-12-31 22:00:00  15372         -2.2               52            0.0   
2025-12-31 22:00:00  14834         -2.2               52            0.0   
2025-12-31 23:00:00  15113         -1.9               47            0.0   
2025-12-31 23:00:00  14248         -1.9               47            0.0   

                     Wind_Boston  Temp_Hartford  Humidity_Hartford  \
2018-01-07 23:00:00          

In [25]:
print(df.head())
print(df.dtypes)

                      load  Temp_Boston  Humidity_Boston  Precip_Boston  \
2018-01-07 23:00:00  15542        -15.1               63            0.0   
2018-01-08 00:00:00  17627        -14.0               60            0.0   
2018-01-08 00:00:00  14818        -14.0               60            0.0   
2018-01-08 01:00:00  17553        -13.2               60            0.0   
2018-01-08 01:00:00  14408        -13.2               60            0.0   

                     Wind_Boston  Temp_Hartford  Humidity_Hartford  \
2018-01-07 23:00:00          9.4          -14.1                 70   
2018-01-08 00:00:00         10.9          -14.0                 68   
2018-01-08 00:00:00         10.9          -14.0                 68   
2018-01-08 01:00:00         14.5          -14.1                 71   
2018-01-08 01:00:00         14.5          -14.1                 71   

                     Precip_Hartford  Wind_Hartford  Temp_Providence  ...  \
2018-01-07 23:00:00              0.0            9.2